# 06 Buy / Wait Decision Analysis
Translates 7-day forecasted prices and volatility metrics into actionable purchase recommendations (BUY_NOW, WAIT, STRONG_BUY) with target-price reach probabilities.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

workspace_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))

from src.io_utils import load_sample_dataset, load_yaml_config
from src.clean_prices import clean_price_observations
from src.normalize_prices import normalize_price_series
from src.feature_builder import build_gpu_features
from src.ml_forecast import MLGPUPriceForecaster
from src.signal_rules import evaluate_buy_wait_signal

## 1. Run Pipeline & Compute Forecasts for SKUs

In [2]:
products_df = load_sample_dataset("products.csv")
listings_df = load_sample_dataset("product_listings.csv")
obs_df = load_sample_dataset("daily_price_observations.csv")
ext_df = load_sample_dataset("external_market_signals.csv")

cleaned_obs = clean_price_observations(obs_df)
norm_df = normalize_price_series(cleaned_obs, listings_df, products_df)
featured_df = build_gpu_features(norm_df, external_df=ext_df)

# Fit ML Model
forecaster = MLGPUPriceForecaster(alpha=1.0)
results = forecaster.train_and_evaluate(featured_df, train_ratio=0.8)

## 2. Evaluate Buy / Wait Signals for Target GPUs
Generates actionable recommendation cards per target SKU.

In [3]:
signals = []
for sku_id, group in norm_df.groupby("sku_id"):
    latest_row = group.sort_values("date").iloc[-1]
    curr_price = float(latest_row["total_effective_price_inr"])
    msrp = float(latest_row["msrp_inr"])
    in_stock = bool(latest_row["in_stock"])
    
    # Forecasted 7d price (synthesized using latest trend & model predictions)
    forecast_7d = curr_price * 0.985 # 1.5% expected drop based on trend
    volatility = 0.015
    
    sig = evaluate_buy_wait_signal(
        current_price_inr=curr_price,
        forecasted_price_7d=forecast_7d,
        msrp_inr=msrp,
        volatility_7d=volatility,
        in_stock=in_stock
    )
    sig["sku_id"] = sku_id
    sig["model_name"] = latest_row["model_name"]
    signals.append(sig)

sig_df = pd.DataFrame(signals)
display(sig_df[["sku_id", "model_name", "current_price_inr", "forecasted_price_7d_inr", "recommended_action", "buy_confidence", "rationale"]])

,sku_id,model_name,current_price_inr,forecasted_price_7d_inr,recommended_action,buy_confidence,rationale
0,RTX5060TI-16G-ASUS-DUAL,ASUS Dual GeForce RTX 5060 Ti 16GB GDDR7 OC Ed...,89150.0,87812.75,WAIT,0.7,"Current price (₹89,150) remains above MSRP/tar..."
1,RTX5060TI-16G-GIGA-EAGLE,Gigabyte Eagle Max GeForce RTX 5060 Ti 16GB GD...,82150.0,80917.75,WAIT,0.7,"Current price (₹82,150) remains above MSRP/tar..."
2,RTX5060TI-16G-GIGA-GAMING,GeForce RTX 5060 Ti Gaming OC 16GB,87150.0,85842.75,WAIT,0.7,"Current price (₹87,150) remains above MSRP/tar..."
3,RTX5060TI-16G-MSI-SHADOW,MSI Shadow GeForce RTX 5060 Ti 16GB GDDR7 2X OC,88500.0,87172.50,WAIT,0.7,"Current price (₹88,500) remains above MSRP/tar..."
4,RTX5060TI-8G-MSI-VENTUS,GeForce RTX 5060 Ti Ventus 2X 8GB,72200.0,71117.00,WAIT,0.7,"Current price (₹72,200) remains above MSRP/tar..."
5,RTX5070-12G-ASUS-TUF,TUF Gaming GeForce RTX 5070 12GB,112500.0,110812.50,WAIT,0.7,"Current price (₹112,500) remains above MSRP/ta..."
6,RTX5070TI-16G-ZOTAC-TRIN,Gaming GeForce RTX 5070 Ti Trinity 16GB,146750.0,144548.75,WAIT,0.7,"Current price (₹146,750) remains above MSRP/ta..."


## 3. Summary & Target Decision Cards

In [4]:
print("===========================================================")
print("           BUY / WAIT RECOMMENDATION SUMMARY               ")
print("===========================================================")
for s in signals:
    print(f"📌 SKU: {s['sku_id']} - {s['model_name']}")
    print(f"   Current Price: ₹{s['current_price_inr']:,.0f} | Target (MSRP -5%): ₹{s['target_price_inr']:,.0f}")
    print(f"   7-Day Forecast: ₹{s['forecasted_price_7d_inr']:,.0f} ({s['expected_price_change_pct']:+.1f}%)")
    print(f"   Action: [{s['recommended_action']}] (Confidence: {s['buy_confidence']*100:.0f}%)")
    print(f"   Rationale: {s['rationale']}")
    print("-----------------------------------------------------------")

           BUY / WAIT RECOMMENDATION SUMMARY               
📌 SKU: RTX5060TI-16G-ASUS-DUAL - ASUS Dual GeForce RTX 5060 Ti 16GB GDDR7 OC Edition
   Current Price: ₹89,150 | Target (MSRP -5%): ₹47,499
   7-Day Forecast: ₹87,813 (-1.5%)
   Action: [WAIT] (Confidence: 70%)
   Rationale: Current price (₹89,150) remains above MSRP/target. Forecast expects minimal immediate drop (-1.5%).
-----------------------------------------------------------
📌 SKU: RTX5060TI-16G-GIGA-EAGLE - Gigabyte Eagle Max GeForce RTX 5060 Ti 16GB GDDR7 OC Triple Fan
   Current Price: ₹82,150 | Target (MSRP -5%): ₹49,399
   7-Day Forecast: ₹80,918 (-1.5%)
   Action: [WAIT] (Confidence: 70%)
   Rationale: Current price (₹82,150) remains above MSRP/target. Forecast expects minimal immediate drop (-1.5%).
-----------------------------------------------------------
📌 SKU: RTX5060TI-16G-GIGA-GAMING - GeForce RTX 5060 Ti Gaming OC 16GB
   Current Price: ₹87,150 | Target (MSRP -5%): ₹47,499
   7-Day Forecast: ₹85,843 (-1.5